# Editing the MEI file live

Session 3, *Linking and editing*: the start of the session.

At the end of session 2 we saw Copilot add a slur to our Schumann file while the score was redrawn next to it. Now you do it yourself. You edit a copy of the starting file in the editor, save it, and the notebook draws the score again. First you make one edit by hand, then one with Copilot. After each edit, you check what changed and compare it with the print.

Our starting file has no slurs, dynamics or tempo words: we removed them on purpose, so that you can restore them from the print.

Keep this notebook in the folder `02-building-an-edition`, next to `schumann-edition-inputs.zip` (`get_notebooks.py` puts both there). It works in the same folder `schumann_edition_work` as `02_pdf_preparation.ipynb` from session 2.

**How to use this notebook:** run the cells from top to bottom with Shift+Enter. If something goes wrong, the message appears under the cell.

## Setup

The first cell makes sure the packages are installed; after the setup in session 2 they already are, so it finishes quickly. The second cell defines helper functions; it is collapsed because you do not need to read it. The third makes your working copy: `schumann_edition_work/edition/mei/work.mei`. This is the file you edit today. If it is already there, the cell keeps it as it is, with your edits.

In [ ]:
%pip install -q "camat[validation,notebooks]==0.2.8" verovio==6.3.0

In [ ]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

"""Live editing of the Schumann MEI file: working copy, print excerpt, change report, check."""

import copy
import difflib
import hashlib
import io
import json
import subprocess
import sys
import zipfile
from pathlib import Path
import xml.etree.ElementTree as ET

from IPython.display import Image, display

MEI_NS = "http://www.music-encoding.org/ns/mei"
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"
ET.register_namespace("", MEI_NS)
INPUTS_ZIP = "schumann-edition-inputs.zip"
TASK_ELEMENTS = {"dynam", "tempo", "slur", "annot"}  # what the tasks in this notebook add
IMSLP_PAGE = "https://imslp.org/wiki/Special:ImagefromIndex/51556"


def _name(element):
    return element.tag.split("}")[-1]


def bar_name(n):
    """MEI numbers the pickup as measure 1; musicians call the next bar bar 1."""
    if n == "1":
        return "the pickup"
    return f"bar {int(n) - 1}" if n and n.isdigit() else f"measure {n}"


def _count(n, word):
    return f"{n} {word}" + ("" if n == 1 else "s")


def _pitch(note):
    accidental = {"f": "♭", "s": "♯", "n": "", "ff": "𝄫", "ss": "𝄪"}
    sign = accidental.get(note.get("accid") or note.get("accid.ges") or "n", "")
    return f"{note.get('pname', '?').upper()}{sign}{note.get('oct', '')}"


# --- The working file -------------------------------------------------------

def _find_inputs_zip(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        for candidate in (folder / INPUTS_ZIP, folder / "notebooks" / INPUTS_ZIP,
                          folder / "sessions/02-building-an-edition/notebooks" / INPUTS_ZIP):
            if candidate.is_file():
                return candidate
    return None


def _starter_bytes(project):
    bundle = _find_inputs_zip()
    if bundle:
        with zipfile.ZipFile(bundle) as archive:
            data = archive.read("starter.mei")
            expected = json.loads(archive.read("manifest.json"))["files_sha256"]["starter.mei"]
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError(f"starter.mei in {bundle} is damaged. Download the ZIP again with get_notebooks.py.")
        return data
    unpacked = sorted((project / "inputs").glob("*/starter.mei"))
    if unpacked:
        return unpacked[-1].read_bytes()
    raise FileNotFoundError(f"Put {INPUTS_ZIP} in the same folder as this notebook "
                            "(get_notebooks.py does this), then run this cell again.")


def _write_new(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as output:
        output.write(data)


def prepare_work_file(project):
    """Return (work, before). Creates each file once and never overwrites one."""
    project = Path(project).resolve()
    work = project / "edition/mei/work.mei"
    before = project / "edition/checkpoints/before-editing.mei"
    linked = project / "edition/checkpoints/linked-before-editing.mei"
    if not before.exists():
        if linked.exists():
            before = linked  # the linking notebook ran first: keep its page links
        else:
            _write_new(before, _starter_bytes(project))
    if work.exists():
        print("Your working file is already there. It is kept as it is:")
    else:
        _write_new(work, before.read_bytes())
        print("Copied the starting file to your working file:")
    print("  ", work.relative_to(project.parent))
    print("The file as it was before editing (do not edit this one):")
    print("  ", before.relative_to(project.parent))
    return work, before


# --- The print --------------------------------------------------------------

def show_print(project, width=1000):
    """Show the first two systems of page 8, from 02_pdf_preparation's page image."""
    image = Path(project) / "edition/pages/page-0001.png"
    if not image.is_file():
        print("The page image is not there yet. 02_pdf_preparation.ipynb makes it (session 2, part 6):\n"
              "run that notebook to the end, then run this cell again.\n"
              f"Until then, look at page 8 of the scan on IMSLP: {IMSLP_PAGE}")
        return
    from PIL import Image as PILImage
    with PILImage.open(image) as page:
        w, h = page.size
        # Systems 1 and 2 of the 300 dpi page (4167 x 5617 pixels), without the title.
        crop = page.crop((round(w * 0.036), round(h * 0.100), round(w * 0.962), round(h * 0.410)))
        crop.thumbnail((width, width))
        buffer = io.BytesIO()
        crop.convert("RGB").save(buffer, format="PNG")
    display(Image(data=buffer.getvalue()))


# --- Reading the file -------------------------------------------------------

def _read(path):
    """Parse an MEI file; on an XML error print what to do and return None."""
    try:
        return ET.parse(path).getroot()
    except ET.ParseError as error:
        line, column = error.position
        print(f"{Path(path).name} cannot be read as XML: the problem is at line {line}, column {column}.\n"
              "Usually a tag is not closed, or a < or a quotation mark is missing, at or just before that line.\n"
              "Undo your last change in the editor (Ctrl+Z, on a Mac Cmd+Z), save, and run this cell again.")
        return None


def _measures(root):
    """Map every element inside a measure to that measure."""
    owner = {}
    for measure in root.iter(f"{{{MEI_NS}}}measure"):
        for element in measure.iter():
            owner[element] = measure
    return owner


def measure_xml(root, n):
    """The XML text of measure @n, as in the file."""
    for measure in root.iter(f"{{{MEI_NS}}}measure"):
        if measure.get("n") == str(n):
            excerpt = copy.deepcopy(measure)
            excerpt.tail = None
            lines = ET.tostring(excerpt, encoding="unicode").replace(f' xmlns="{MEI_NS}"', "").split("\n")
            indent = min((len(line) - len(line.lstrip()) for line in lines[1:] if line.strip()), default=0)
            return "\n".join([lines[0]] + [line[indent:] for line in lines[1:]])
    raise ValueError(f"No measure n=\"{n}\" in the file.")


def slur_prompt(work):
    root = _read(work)
    if root is None:
        return ""
    return f'''I am editing an MEI file (MEI 5.1) of Schumann, Album für die Jugend, Op. 68 No. 10.
Add the slur printed in the left hand (staff 2). It starts on the C3 of the pickup,
measure n="1", and ends on the last note of the next measure, n="2", an F3.
It curves above the notes.

Add one <slur> element inside the pickup measure, after its <staff> elements, with
startid and endid pointing to the xml:id of the two notes (with a # in front),
staff="2", curvedir="above" and xml:id="seminar-slur-pickup". Change nothing else.

Here are the two measures from the file:

{measure_xml(root, 1)}
{measure_xml(root, 2)}
'''


# --- What changed -----------------------------------------------------------

def _key(element):
    attributes = tuple(sorted((k, v) for k, v in element.attrib.items() if k != XML_ID))
    return _name(element), element.get(XML_ID), attributes, (element.text or "").strip()


def _describe(element, by_id, owner):
    """One line for an element: its name, text, notes it points to, other attributes."""
    name = _name(element)
    parts = [f"<{name}>"]
    if name == "note":
        parts.append(_pitch(element))
    text = " ".join("".join(element.itertext()).split())
    if text:
        parts.append(f'"{text[:60]}"')
    if element.get(XML_ID):
        parts.append(f"(xml:id {element.get(XML_ID)})")
    children = list(element)
    if name in ("beam", "tuplet") and children:
        kinds = sorted({_name(child) for child in children})
        ids = ", ".join(child.get(XML_ID) for child in children if child.get(XML_ID))
        parts.append(f"around {_count(len(children), kinds[0] if len(kinds) == 1 else 'element')}"
                     + (f" ({ids})" if ids else ""))
    details = []
    for attribute in ("startid", "endid"):
        target = (element.get(attribute) or "").lstrip("#")
        if target:
            word = "from" if attribute == "startid" else "to"
            note = by_id.get(target)
            if note is None:
                details.append(f"{word} #{target} (no element in the file has this xml:id!)")
            else:
                where = bar_name(owner[note].get("n")) if note in owner else "outside the bars"
                pitch = _pitch(note) if _name(note) == "note" else f"<{_name(note)}>"
                details.append(f"{word} {pitch} in {where}")
    for attribute, value in sorted(element.attrib.items()):
        if attribute not in (XML_ID, "startid", "endid") and "}" not in attribute:
            details.append(f'{attribute}="{value}"')
    return " ".join(parts) + (": " + ", ".join(details) if details else "")


def show_changes(before, work):
    """Print, in plain words, what changed in work.mei since the file before editing.

    Spaces and line breaks are not counted. This never stops the notebook.
    """
    old, new = _read(before), _read(work)
    if old is None or new is None:
        return
    old_elements, new_elements = list(old.iter()), list(new.iter())
    old_owner, new_owner = _measures(old), _measures(new)
    new_ids = {}
    duplicates = set()
    for element in new_elements:
        if element.get(XML_ID):
            if element.get(XML_ID) in new_ids:
                duplicates.add(element.get(XML_ID))
            new_ids[element.get(XML_ID)] = element
    old_ids = {e.get(XML_ID): e for e in old_elements if e.get(XML_ID)}

    changes = []  # (where, sign, text, asked_for)
    matcher = difflib.SequenceMatcher(None, [_key(e) for e in old_elements],
                                      [_key(e) for e in new_elements], autojunk=False)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        removed, added = old_elements[i1:i2], new_elements[j1:j2]
        same_id = {e.get(XML_ID) for e in removed if e.get(XML_ID)} & {e.get(XML_ID) for e in added}
        for element in added:
            where = new_owner.get(element)
            where = bar_name(where.get("n")) if where is not None else "outside the bars"
            if element.get(XML_ID) in same_id:
                was = next(e for e in removed if e.get(XML_ID) == element.get(XML_ID))
                diffs = [f'{a}: "{was.get(a)}" → "{element.get(a)}"' for a in sorted(set(was.attrib) | set(element.attrib))
                         if was.get(a) != element.get(a)]
                if (was.text or "").strip() != (element.text or "").strip():
                    diffs.append(f'text: "{(was.text or "").strip()}" → "{(element.text or "").strip()}"')
                label = f"<{_name(element)}> {element.get(XML_ID)}"
                if _name(element) == "note":
                    label += f" ({_pitch(was)} → {_pitch(element)})" if _pitch(was) != _pitch(element) else f" ({_pitch(element)})"
                changes.append((where, "~", f"changed {label}: " + "; ".join(diffs), False))
            elif not any(element in parent for parent in added if parent is not element):
                asked = _name(element) in TASK_ELEMENTS and element.get(XML_ID) not in old_ids
                changes.append((where, "+", "new " + _describe(element, new_ids, new_owner), asked))
        for element in removed:
            if element.get(XML_ID) in same_id or any(element in parent for parent in removed if parent is not element):
                continue
            where = old_owner.get(element)
            where = bar_name(where.get("n")) if where is not None else "outside the bars"
            changes.append((where, "-", "removed " + _describe(element, old_ids, old_owner), False))

    if not changes:
        print(f"No changes yet: {Path(work).name} is the same as before editing (spaces and line breaks are not counted).")
        return
    print(f"Changes in {Path(work).name} since before editing (spaces and line breaks are not counted):")
    places = []
    for where, *_ in changes:
        if where not in places:
            places.append(where)
    for where in places:
        print(f"\n{where[0].upper() + where[1:]}")
        for _, sign, text, asked in (c for c in changes if c[0] == where):
            print(f"  {sign} {text}" + ("" if asked else "   ← not part of the tasks"))
    problems = [f"xml:id {i} is used more than once" for i in sorted(duplicates)]
    for element in new_elements:
        for attribute in ("startid", "endid", "plist"):
            for token in (element.get(attribute) or "").split():
                if token.startswith("#") and token[1:] not in new_ids:
                    problems.append(f"<{_name(element)}> {attribute} points to {token}, but no element has that xml:id")
    others = [text for _, sign, text, asked in changes if not asked]
    if others:
        print(f"\n{_count(len(others), 'change')} {'was' if len(others) == 1 else 'were'} not part of the tasks. "
              "Look at each one in the print: keep it if the print shows it, otherwise undo it in the editor.")
    if problems:
        print("\nTo fix:")
        for problem in problems:
            print("  ✗", problem)


# --- Checking the file ------------------------------------------------------

def _validate(path, folder):
    result = subprocess.run([sys.executable, "-m", "camat.mei_validation", str(path),
                             "--output-dir", str(folder), "--verovio"],
                            text=True, capture_output=True, timeout=300)
    findings_file = Path(folder) / "findings.json"
    if not findings_file.is_file():
        print("The checker did not run:")
        print(result.stderr[-2000:])
        return None
    return json.loads(findings_file.read_text())


def _finding_key(finding):
    # Line numbers move when lines are added, so they are not part of the comparison.
    return tuple(finding.get(k) for k in ("severity", "check", "element", "xml_id", "measure_n", "staff_n", "layer_n", "message"))


def check_file(work, before, project):
    """Run CAMAT's MEI checker on work.mei and report what is new since before editing."""
    if _read(work) is None:
        return
    project = Path(project)
    now = _validate(work, project / "edition/validation-editing")
    then = _validate(before, project / "edition/validation-before-editing")
    if now is None or then is None:
        return
    known = {_finding_key(f) for f in then}
    new = [f for f in now if _finding_key(f) not in known]
    errors = [f for f in new if f["severity"] == "error"]
    print(f"Checked {Path(work).name} against the MEI rules (schema) and with Verovio.")
    if not new:
        print(f"✓ No new problems. (The file before editing already had {_count(len(then), 'remark')}; they are not shown.)")
        return
    if not errors:
        print("✓ The file follows the MEI rules.")
    for finding in new:
        where = bar_name(finding["measure_n"]) if finding.get("measure_n") else "outside the bars"
        mark = "✗" if finding["severity"] == "error" else "!"
        print(f"  {mark} line {finding.get('line') or '?'} ({where}): {finding['message']}")
    if errors:
        print("Fix the lines marked ✗, save, and run this cell again.")
    print(f"The full report is in {(project / 'edition/validation-editing').relative_to(project.parent)}.")

In [ ]:
PROJECT = Path.cwd() / "schumann_edition_work"
WORK_MEI, BEFORE_MEI = prepare_work_file(PROJECT)

<details><summary>How do I start again from the beginning?</summary>

Delete `work.mei` (or rename it, to keep it), then run the cell above again. It makes a new copy from the file as it was before editing.

</details>

## 1. The print

The next cell shows the first two systems of page 8 of our print. It uses the page image that `02_pdf_preparation.ipynb` made in session 2. Look for what our file is missing:

- **Frisch und munter.** above the first bar: the tempo words;
- **f** under the right hand at the start: the dynamic, *forte*;
- the long curved line in the left hand, from the pickup to the end of bar 1, and a second one in bar 2: two slurs.

Remember how we count: the short bar at the start is "the pickup", and bar 1 is the first full bar. The MEI file numbers the pickup as measure 1, so bar 1 is `<measure n="2">`.

In [ ]:
show_print(PROJECT)

## 2. Start the live score

Run the next cell. It draws the score from `work.mei`. Because **Auto-watch MEI** is switched on, it draws the score again each time you save the file. If you save and nothing changes, click **Reload score**.

In [ ]:
from camat import launch_interactive_facsimile_viewer

if "viewer" in globals():
    viewer.stop_watch()
viewer = launch_interactive_facsimile_viewer(
    str(WORK_MEI), auto_watch_mei=True,
    verovio_breaks="encoded" if "<sb" in WORK_MEI.read_text(encoding="utf-8") else "auto",
    verovio_options={"header": "none", "footer": "none"},
)

Now open `work.mei` in the editor, so that you see the file and the score at the same time:

- **VS Code:** in the file list on the left, open `schumann_edition_work` → `edition` → `mei` and click `work.mei`. Drag its tab to the right half of the window to see it beside the notebook.
- **JupyterLab (cloud route):** in the file browser, open `schumann_edition_work` → `edition` → `mei` and double-click `work.mei`; it opens as text. Drag its tab to the right edge of the window.

## 3. An edit by hand: the opening marks

1. In `work.mei`, search for `"d1e9"` with the quotation marks (Ctrl+F, on a Mac Cmd+F). You land on `<measure n="1" xml:id="d1e9" ...>`: the pickup.
2. The pickup has two `<staff>` elements, one for each hand. Put the cursor at the start of the line with `</measure>` below them, and paste these two lines:

   ```xml
   <dynam xml:id="seminar-dynam-opening" staff="1" place="below" tstamp="1">f</dynam>
   <tempo xml:id="seminar-tempo-opening" staff="1" place="above" tstamp="1">Frisch und munter.</tempo>
   ```

3. Save the file (Ctrl+S, on a Mac Cmd+S). The score above is drawn again, with the *f* and the tempo words. Compare it with the print.

<details><summary>What do the attributes mean?</summary>

`staff="1"` attaches the mark to the upper staff (the right hand), `place` puts it below or above that staff, and `tstamp="1"` puts it on the first beat of the measure. `xml:id` gives the new element a name of its own, so that other parts of the edition can point at it, just as the notes have names such as `d1e60`.

</details>

Then run the next cell. It lists what changed in `work.mei` since before editing, in words.

In [ ]:
show_changes(BEFORE_MEI, WORK_MEI)

You should see two new elements in the pickup: a `<dynam>` and a `<tempo>`.

<details><summary>The score is not drawn again, or the cell shows an error</summary>

Most likely a character went missing while pasting, for example a `<` or a quotation mark, and the file is no longer correct XML. The cell above tells you the line where the problem is. Undo your last change in the editor (Ctrl+Z, on a Mac Cmd+Z), save, and paste again.

</details>

## 4. An edit with Copilot: the slur

Now let a language model make an edit: the long slur in the left hand, from the C3 of the pickup to the F3 at the end of bar 1. The next cell writes the request (a *prompt*) for you, with the two measures from your file.

In [ ]:
print(slur_prompt(WORK_MEI))

1. Open Copilot's chat in VS Code with **View → Chat** (setup, step 6). On the cloud route, or without Copilot, use any chat assistant in your browser.
2. Copy the prompt from the cell above into the chat and send it.
3. If Copilot offers to change `work.mei` directly, look at the change and accept it. If it answers with a line of XML, paste that line into the pickup measure yourself, after the `<dynam>` and `<tempo>` from part 3.
4. Save the file and look at the score: is the slur there, and does it look like the one in the print?

Then run the next cell.

In [ ]:
show_changes(BEFORE_MEI, WORK_MEI)

Read the new `<slur>` line: it names the two notes the slur connects, with their pitch and bar. They should be the C3 in the pickup and the F3 in bar 1. If the model picked other notes, correct `startid` or `endid` by hand.

Language models often change more than you asked for. Lines marked **not part of the tasks** show such changes. In class, Copilot also added a `<beam>` around two chords in bar 1. Look at the print: those two chords are indeed joined by a beam, so the change is right, but nobody asked for it. For each change like this, look at the print and decide: keep it, or undo it.

### Extensions

- **The second slur.** The print has another slur in the left hand of bar 2. Add it by hand or with Copilot, with a new `xml:id` such as `seminar-slur-bar2`. Which notes does it connect?
- **The second *f*.** The second system begins with *f* again (bar 5, `<measure n="6">`). Add it as in part 3, with the new `xml:id` `seminar-dynam-second-system`.
- **Two assistants.** Give the same prompt to a second chat assistant. Do the two answers choose the same notes? Do they change anything else?

## 5. Check the file

The score looks right, but is the file still a correct MEI file? The next cell checks `work.mei` against the MEI rules (the *schema*) and draws it with Verovio. It also checks the file as it was before editing, and shows only what is new: the starting file already has a few remarks, for example that the pickup is shorter than a full bar.

In [ ]:
check_file(WORK_MEI, BEFORE_MEI, PROJECT)

## At the end

Your edits stay in `work.mei`; we continue with this file. Run the last cell to stop the live score from watching the file.

In [ ]:
viewer.stop_watch()